In [1]:
import pandas as pd
import numpy as np
import re
import os
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import (LightningDataModule, LightningModule, Trainer)
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer
from pytorch_lightning.loggers import WandbLogger
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error 
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATA_FOLDER = "./"
from sklearn.preprocessing import StandardScaler, RobustScaler

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_big_best_customers_5c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
df = df[df["product_id"].isin(product_ids)]
# sort df by date
df = df.sort_values(by=["date_id"])
target = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)
target_scaler = RobustScaler(with_centering=False).fit(target.dropna().values.reshape(-1, 1))
df["target"] = target

# entreno en todos los df donde target no es NaN
train = df[df["target"].notna()]
train["target"] = target_scaler.transform(train["target"].values.reshape(-1, 1)).flatten()
train["weights"] = np.log1p(train["tn"]).clip(1, 10)
# valido en date_id == 33
val = train[train["date_id"] == 33]

scaler = RobustScaler(with_centering=False)
train[numeric_cols] = scaler.fit_transform(train[numeric_cols].fillna(0))
val[numeric_cols] = scaler.transform(val[numeric_cols].fillna(0))


X_kaggle = df[df["date_id"] == 35][numeric_cols].fillna(0)

# scalo X_train, X_kaggle y X_val con estandar scaler
X_kaggle[numeric_cols] = scaler.transform(X_kaggle[numeric_cols])

target_col = "target"
weight_name = "weights"


/tmp/ipykernel_725501/284001704.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = target
/tmp/ipykernel_725501/284001704.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train["target"] = target_scaler.transform(train["target"].values.reshape(-1, 1)).flatten()
/tmp/ipykernel_725501/284001704.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(a

In [3]:
class custom_args():
    def __init__(self):
        self.usegpu = True
        self.gpuid = 0
        self.seed = 42
        self.model = "nn"
        self.loader_workers = 0
        self.bs = 8192
        self.lr = 1e-3
        self.weight_decay = 5e-4
        self.droputs = [0.1, 0.1]
        self.n_hidden = [512, 512, 256]
        self.patience = 100
        #self.max_epochs = 20
        self.max_epochs = 8000
        self.N_fold = 15

my_args = custom_args()

In [4]:
#Pytorch Data Module Definition
from pytorch_lightning.utilities.types import EVAL_DATALOADERS


class CustomDataset(Dataset):
    def __init__(self, df, accelerator):
        self.features = torch.FloatTensor(df[numeric_cols].values).to(accelerator)
        self.labels = torch.FloatTensor(df[target_col].values).to(accelerator)
        self.weights = torch.FloatTensor(df[weight_name].values).to(accelerator)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        w = self.weights[idx]
        return x, y, w
    
class DataModule(LightningDataModule):
    def __init__(self, train_df, batch_size, valid_df=None, accelerator="cpu"):
        super().__init__()
        self.df = train_df
        self.batch_size = batch_size
        self.dates = self.df["date_id"].unique()
        self.accelerator = accelerator
        self.train_dataset = None
        self.valid_df = None
        if valid_df is not None:
            self.valid_df = valid_df
        self.val_dataset = None

    def setup(self, fold=0, N_fold=5, stage=None):
        selected_dates = [date for ii, date in enumerate(self.dates) if ii % N_fold != fold]
        df_train = self.df.loc[self.df["date_id"].isin(selected_dates)]
        self.train_dataset = CustomDataset(df_train, self.accelerator)
        if self.valid_df is not None:
            df_valid = self.valid_df
            self.val_dataset = CustomDataset(df_valid, self.accelerator)

    def train_dataloader(self, n_workers=0):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=n_workers,
        )
    
    def val_dataloader(self, n_workers=0):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=n_workers,
        )
    
class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, droputs, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))
            if i > 0:
                layers.append(nn.SiLU())
            if i < len(droputs):
                layers.append(nn.Dropout(droputs[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        layers.append(nn.ReLU())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        return self.model(x).squeeze(-1)
    
    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none')
        loss = (loss * w).mean()
        self.log("train_loss", loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss
    
    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none')
        loss = (loss * w).mean()
        self.validation_step_outputs.append((y_hat, y, w))
        self.log("val_loss", loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }
    
    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

In [5]:
# crete pytorch data module
args = my_args
device = torch.device("cuda" if args.usegpu and torch.cuda.is_available() else "cpu")
accelerator = "cuda" if args.usegpu and torch.cuda.is_available() else "cpu"
loader_device = "cpu"

data_module = DataModule(
    train_df=train,
    batch_size=args.bs,
    valid_df=val,
    accelerator=accelerator
)


In [6]:
import gc
import os

os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"
del df
gc.collect()
pl.seed_everything(args.seed)
models = []
for fold in range(args.N_fold):
    data_module.setup(fold=fold, N_fold=args.N_fold)
    input_dim = data_module.train_dataset.features.shape[1]
    model = NN(
        input_dim=input_dim,
        hidden_dims=args.n_hidden,
        droputs=args.droputs,
        lr=args.lr,
        weight_decay=args.weight_decay
    )
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=args.patience,
        mode="min",
        verbose=False,
    )
    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        verbose=False,
        dirpath="./models",           # Carpeta donde querés guardar los checkpoints
        filename=f"./nn_{fold}.model"
    )
    timer = Timer()
    trainer = Trainer(
        max_epochs=args.max_epochs,
        accelerator=accelerator,
        devices=[args.gpuid] if args.usegpu else None,
        callbacks=[early_stopping, checkpoint_callback, timer],
        enable_progress_bar=True,
    )
    trainer.fit(model, data_module.train_dataloader(args.loader_workers), data_module.val_dataloader(args.loader_workers))
    print(f'Fold-{fold} Training completed in {timer.time_elapsed("train"):.2f}s')
    models.append(model)



Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 732 K  | train
---------------------------------------------
732 K     Trainable params
0         Non-trainable params
732 K     Total params
2.930     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (14) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 219: 100%|██████████| 14/14 [00:01<00:00, 10.05it/s, v_num=57]
Fold-0 Training completed in 310.61s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/fede/programacion/labo3/models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 732 K  | train
---------------------------------------------
732 K     Trainable params
0         Non-trainable params
732 K     Total params
2.930     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/fede/programacion/labo3/models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 732 K  | train
---------------------------------------------
732 K     Trainable params
0         Non-trainable params
732 K     Total params
2.930     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (15) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 183: 100%|██████████| 15/15 [00:01<00:00,  9.72it/s, v_num=71]
Fold-14 Training completed in 283.64s


# inference

In [7]:
models = []
for fold in range(args.N_fold):
    model_path = f"./models/nn_{fold}.model.ckpt"
    if os.path.exists(model_path):
        model = NN.load_from_checkpoint(model_path)
        models.append(model.to("cuda:0"))
    else:
        print(f"Model for fold {fold} not found at {model_path}")
models


[NN(
   (model): Sequential(
     (0): BatchNorm1d(653, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (1): Dropout(p=0.1, inplace=False)
     (2): Linear(in_features=653, out_features=512, bias=True)
     (3): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (4): SiLU()
     (5): Dropout(p=0.1, inplace=False)
     (6): Linear(in_features=512, out_features=512, bias=True)
     (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (8): SiLU()
     (9): Linear(in_features=512, out_features=256, bias=True)
     (10): Linear(in_features=256, out_features=1, bias=True)
     (11): ReLU()
   )
 ),
 NN(
   (model): Sequential(
     (0): BatchNorm1d(653, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (1): Dropout(p=0.1, inplace=False)
     (2): Linear(in_features=653, out_features=512, bias=True)
     (3): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stat

In [8]:
X_val = val[numeric_cols]
y_val = val[target_col]


y_pred_valid = np.zeros(y_val.shape)
with torch.no_grad():
    for model in models:
        model.eval()
        y_pred_valid += model(torch.FloatTensor(X_val.values).to("cuda:0")).cpu().numpy() / len(models)

y_pred_valid

array([0.04663732, 0.        , 0.        , ..., 0.        , 0.06433796,
       3.63920894])

In [9]:
print(list(y_pred_valid))

[np.float64(0.04663731902837753), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.8077956605702639), np.float64(0.0), np.float64(0.15172338299453259), np.float64(0.001462658285163343), np.float64(0.036024060100317), np.float64(15.276416003704071), np.float64(0.5616184351965785), np.float64(0.009599019773304462), np.float64(0.5854467707686126), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(9.350014328956604), np.float64(0.08625137805938721), np.float64(3.3428944647312164), np.float64(3.7762775868177414), np.float64(2.5969161838293076), np.float64(27.933611631393433), np.float64(4.625479444861412), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(22.834051251411438), np.float64(1.8861752897500992), np.float64(0.0), np.float64(0.0), np.float64(0.8323964029550552), np.float64(1.7381983324885368), np.float64(0.0), np.float64(0.03380203898996115),

In [10]:
target_scaler.inverse_transform(y_pred_valid.reshape(-1, 1)).flatten()

array([ 0.13469465,  0.        ,  0.        , ...,  0.        ,
        0.1858164 , 10.51050892])

In [11]:
test_df = val.copy()
test_df[numeric_cols] = scaler.inverse_transform(test_df[numeric_cols])
test_df["target"] = target_scaler.inverse_transform(test_df["target"].values.reshape(-1, 1)).flatten()
y_pred_valid = target_scaler.inverse_transform(y_pred_valid.reshape(-1, 1)).flatten()
test_df["predictions"] = y_pred_valid
test_df = test_df[["product_id", "customer_id", "date_id", "predictions", "target"]]
test_df =test_df.groupby(["product_id"]).agg({
    "predictions": "sum",
    "target": "sum"
    }
).reset_index()
test_df["abs_error"] = np.abs(test_df["predictions"] - test_df["target"])
test_df

/tmp/ipykernel_725501/3051987149.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df["predictions"] = y_pred_valid


,product_id,predictions,target,abs_error
0,20001.0,1430.882606,1504.688599,73.805993
1,20002.0,1137.594661,1087.308594,50.286067
2,20003.0,884.176061,892.501282,8.325221
3,20004.0,623.427941,637.900024,14.472083
4,20005.0,594.198222,593.244446,0.953776
...,...,...,...,...
775,21263.0,0.000000,0.012700,0.012700
776,21265.0,0.000000,0.050070,0.050070
777,21266.0,0.000000,0.051210,0.051210
778,21267.0,0.000000,0.015690,0.015690


In [12]:
total_error = test_df["abs_error"].sum() / test_df["target"].sum()
total_error

np.float64(0.2245616809949001)

In [13]:

y_pred_valid = np.zeros(X_kaggle.shape[0])
with torch.no_grad():
    for model in models:
        model.eval()
        y_pred_valid += model(torch.FloatTensor(X_kaggle.values).to("cuda:0")).cpu().numpy() / len(models)


submission_df = X_kaggle[["product_id", "customer_id", "date_id"]].copy()
submission_df["predictions"] = y_pred_valid
submission_df[numeric_cols] = scaler.inverse_transform(X_kaggle[numeric_cols])
submission_df["predictions"] = target_scaler.inverse_transform(submission_df["predictions"].values.reshape(-1, 1)).flatten()

/tmp/ipykernel_725501/2996960838.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  submission_df[numeric_cols] = scaler.inverse_transform(X_kaggle[numeric_cols])
/tmp/ipykernel_725501/2996960838.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  submission_df[numeric_cols] = scaler.inverse_transform(X_kaggle[numeric_cols])
/tmp/ipykernel_725501/2996960838.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining 

In [14]:
submission_df

,product_id,customer_id,date_id,predictions,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,sku_size,...,prod_tn_rolling_mean_6_lag_6_x_tn_rolling_mean_24_lag_2,prod_tn_rolling_mean_6_lag_6_x_tn_rolling_mean_24_lag_3,prod_tn_rolling_mean_6_lag_6_x_tn_rolling_mean_6,prod_tn_rolling_mean_6_lag_6_x_tn_rolling_mean_6_lag_8,prod_tn_rolling_mean_24_lag_2_x_tn_rolling_mean_24_lag_3,prod_tn_rolling_mean_24_lag_2_x_tn_rolling_mean_6,prod_tn_rolling_mean_24_lag_2_x_tn_rolling_mean_6_lag_8,prod_tn_rolling_mean_24_lag_3_x_tn_rolling_mean_6,prod_tn_rolling_mean_24_lag_3_x_tn_rolling_mean_6_lag_8,prod_tn_rolling_mean_6_x_tn_rolling_mean_6_lag_8
131775,20743.0,10001.0,35.0,1176.176789,0.0,3.0,0.04512,0.04512,6.71624,200.0,...,0.010383,0.011278,0.002597,0.005265,0.029703,0.006839,0.013865,0.007429,0.015060,0.003468
130835,20737.0,0.0,35.0,1127.950961,0.0,32.0,0.75220,0.75220,13.68429,1000.0,...,1.899130,1.839392,2.575368,0.989984,2.999002,4.198960,1.614100,4.066880,1.563328,2.188846
82859,20424.0,10004.0,35.0,1169.771814,0.0,1.0,0.13135,0.13135,0.13176,100.0,...,0.027197,0.026550,0.022410,0.026916,0.024274,0.020489,0.024608,0.020002,0.024023,0.020277
132380,20746.0,10004.0,35.0,1178.006169,0.0,0.0,0.00000,0.00000,7.72588,350.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
89201,20463.0,10003.0,35.0,1173.517061,0.0,17.0,1.57058,1.57058,4.70588,530.0,...,0.242049,0.252541,0.288872,0.223912,0.290369,0.332141,0.257451,0.346539,0.268611,0.307254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57053,20291.0,10001.0,35.0,1155.859251,0.0,2.0,0.25462,0.25462,15.74343,650.0,...,11.695195,12.132673,6.159146,14.457420,9.768928,4.959192,11.640757,5.144699,12.076199,6.130476
57017,20291.0,0.0,35.0,1683.362073,0.0,67.0,6.03982,6.03982,15.74343,650.0,...,77.211250,77.749779,65.294151,65.346382,91.025429,76.443024,76.504173,76.976196,77.037773,64.696205
146075,20840.0,10005.0,35.0,1167.398699,0.0,0.0,0.00000,0.00000,0.10957,60.0,...,0.007262,0.007830,0.000052,0.011383,0.005938,0.000039,0.008633,0.000042,0.009308,0.000061
57197,20291.0,10005.0,35.0,1179.582311,0.0,0.0,0.00000,0.00000,15.74343,650.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [15]:

submission_df = submission_df[["product_id", "customer_id", "date_id", "predictions"]]
submission_df = submission_df.groupby(["product_id"]).agg({
    "predictions": "sum",
    }
).reset_index()
submission_df

,product_id,predictions
0,20001.0,91211.132247
1,20002.0,72485.276069
2,20003.0,63474.882516
3,20004.0,45919.372663
4,20005.0,41475.987183
...,...,...
775,21263.0,7032.024705
776,21265.0,7039.413882
777,21266.0,7039.239264
778,21267.0,7039.098794


In [16]:
submission_df.columns = ["product_id", "tn"]
submission_df.to_csv("submission_nn.csv", index=False)